# Biostatistics Fundamentals

**Tier 0 -- Computational Foundations | Module 6**

---

## Why Statistics Matters in Bioinformatics

Every major question in modern biology is ultimately a statistical question:

- **Differential expression**: Is this gene significantly up-regulated in tumor vs. normal tissue?
- **Variant calling**: Is this SNP a real mutation or a sequencing error?
- **Enrichment analysis**: Are DNA repair genes overrepresented in our hit list?
- **Clinical trials**: Does this drug improve patient survival?
- **Quality control**: Is this sample an outlier that should be removed?

Without solid statistical reasoning, you cannot distinguish signal from noise. This module provides the statistical foundations you will use throughout the course.

### Learning Objectives

By the end of this module you will be able to:
- Distinguish populations from samples and choose appropriate statistics
- Compute and interpret descriptive statistics for biological data
- Explain and visualize common probability distributions
- Apply the Central Limit Theorem to understand sampling distributions
- Formulate null and alternative hypotheses and interpret p-values correctly
- Choose the right statistical test (parametric vs. non-parametric)
- Apply multiple testing correction (Bonferroni, FDR) -- essential for genomics
- Perform power analysis to plan experiments

## Why this notebook matters

When you run a differential expression analysis, you are not just running code — you are making statistical claims. Understanding what a p-value actually means, why you need multiple testing correction when analyzing 20,000 genes at once, and why a statistically significant result can still be biologically meaningless prevents you from misinterpreting your own results (and misreading published papers). These are the concepts that separate a bioinformatician who produces valid science from one who produces convincing-looking nonsense.


## How to use this notebook
1. Run imports first (cell below the section headers), then run each section top-to-bottom.
2. Focus on the visualizations — the histograms, boxplots, and power curves convey the concepts better than the formulas.
3. The final worked example (Section 11) ties everything together in a simulated RNA-seq analysis — read it carefully before the exercises.
4. For each statistical test, note which assumptions it requires and what happens when those assumptions are violated.


## Common stumbling points

- **P-value is not the probability that the null hypothesis is true**: It is the probability of observing data this extreme *if* the null hypothesis were true. These are completely different statements.
- **Statistical significance ≠ biological significance**: With n = 10,000 samples, even a 0.001-fold change will be "significant." Always look at effect sizes (log2 fold change, Cohen's d) alongside p-values.
- **Multiple testing explosion**: Testing 20,000 genes at α = 0.05 expects 1,000 false positives even with no real effects. Always apply BH/FDR correction in genomics analyses.
- **Paired vs. unpaired tests**: Using an unpaired test on paired data (before/after on same patients) throws away information and loses statistical power. Always match the test to the study design.
- **The Shapiro-Wilk test is not the right way to choose between t-test and Mann-Whitney**: With large n, Shapiro-Wilk rejects normality for trivially small deviations. With small n, it has low power to detect real non-normality. Look at the data visually and use domain knowledge.


In [ ]:
# Core imports for this module
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.power import TTestIndPower

# Reproducibility
np.random.seed(42)

# Plot styling
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['figure.dpi'] = 100


---

## 1. Populations vs. Samples

| Concept | Population | Sample |
|---------|-----------|--------|
| Definition | All possible observations | The subset we actually measure |
| Size | Usually infinite or impractical to measure | Finite and known |
| Goal | Has true **parameters** | Provides **estimates** (statistics) |
| Notation (mean) | mu (population mean) | x-bar (sample mean) |
| Notation (SD) | sigma (population SD) | s (sample SD) |

### Biological examples

| Population | Sample |
|-----------|--------|
| All human genomes that ever existed or will exist | 1000 Genomes Project (~2,500 individuals) |
| All cancer cells in a patient's body | Cells from a needle biopsy |
| All possible measurements of a gene's expression | Your 3 biological replicates |

**Key insight**: We never know the true population parameters. Statistics lets us make rigorous inferences about the population from our sample.

---

## 2. Types of Data

The type of data determines which statistical tests are appropriate.

### Categorical Data
- **Nominal**: No natural ordering. Examples: blood type (A, B, AB, O), mutation type (missense, nonsense, frameshift), tissue type
- **Ordinal**: Has a natural order, but intervals are not equal. Examples: cancer stage (I, II, III, IV), quality score (low/medium/high)

### Numerical Data
- **Discrete**: Countable integers. Examples: read counts, number of mutations, number of exons
- **Continuous**: Any value in a range. Examples: gene expression (TPM), GC content (%), protein concentration

---

## 3. Descriptive Statistics

Before running any test, always look at your data. Descriptive statistics summarize the center and spread of a distribution.

In [ ]:
# Simulated gene expression data with outliers (realistic for biology)
np.random.seed(42)
expression = np.random.normal(loc=10, scale=2, size=100)

# Add a few outliers (e.g., batch effects or highly expressed genes)
expression = np.append(expression, [20, 22, 25])

# Measures of central tendency
print("=== Measures of Central Tendency ===")
print(f"Mean:   {np.mean(expression):.2f}  (sensitive to outliers)")
print(f"Median: {np.median(expression):.2f}  (robust to outliers)")
print(f"Mode:   {stats.mode(np.round(expression), keepdims=True).mode[0]:.1f}  (most frequent value)")

# Measures of spread
print("\n=== Measures of Spread ===")
print(f"Variance (s^2): {np.var(expression, ddof=1):.2f}")
print(f"Std Dev (s):    {np.std(expression, ddof=1):.2f}")
print(f"Range:          {np.ptp(expression):.2f}  (max - min)")

# IQR -- Interquartile Range (robust to outliers)
q25, q75 = np.percentile(expression, [25, 75])
iqr = q75 - q25
print(f"IQR:            {iqr:.2f}  (Q3 - Q1)")
print(f"Q1 (25th pctl): {q25:.2f}")
print(f"Q3 (75th pctl): {q75:.2f}")

# Standard error of the mean
sem = np.std(expression, ddof=1) / np.sqrt(len(expression))
print(f"\nStandard Error of Mean: {sem:.2f}  (SD / sqrt(n))")
print(f"  This tells us how precisely we know the mean.")
print(f"  n = {len(expression)}, so SEM << SD")

In [ ]:
# Why median > mean when outliers are high
print("Effect of outliers:")
print(f"  Without outliers: mean = {np.mean(expression[:100]):.2f}, median = {np.median(expression[:100]):.2f}")
print(f"  With 3 outliers:  mean = {np.mean(expression):.2f}, median = {np.median(expression):.2f}")
print(f"  The mean shifted by {np.mean(expression) - np.mean(expression[:100]):.2f}, "
      f"but median only by {np.median(expression) - np.median(expression[:100]):.2f}")

In [ ]:
# Visualizing distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogram
axes[0].hist(expression, bins=25, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(np.mean(expression), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(expression):.1f}')
axes[0].axvline(np.median(expression), color='green', linestyle='--', linewidth=2, label=f'Median: {np.median(expression):.1f}')
axes[0].set_xlabel('Expression Level')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution with Outliers')
axes[0].legend(fontsize=9)

# Boxplot
bp = axes[1].boxplot(expression, vert=True, patch_artist=True,
                     boxprops=dict(facecolor='lightblue'))
axes[1].set_ylabel('Expression Level')
axes[1].set_title('Boxplot (outliers shown as circles)')

# Q-Q plot (for normality assessment)
stats.probplot(expression, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot (deviation at tails = outliers)')

plt.tight_layout()
plt.show()

### Understanding the Boxplot

```
          o           <- Outliers (beyond 1.5 * IQR from box edge)
          |
    ------+------     <- Upper whisker (last data point within 1.5 * IQR)
    |            |
    |   Q3 (75%) |    <- Top of box
    |            |
    |----Median---|
    |            |
    |   Q1 (25%) |    <- Bottom of box
    |            |
    ------+------     <- Lower whisker
```

The box contains 50% of the data (the IQR). Points beyond 1.5 * IQR from the box edges are flagged as potential outliers.

---

## 4. Probability Distributions

### 4.1 The Normal (Gaussian) Distribution

The most important distribution in statistics, defined by two parameters: mean (mu) and standard deviation (sigma).

**The empirical rule (68-95-99.7)**:
- ~68% of data falls within 1 SD of the mean
- ~95% within 2 SD
- ~99.7% within 3 SD

In [ ]:
# Visualize the normal distribution and the empirical rule
x = np.linspace(-4, 4, 1000)
y = stats.norm.pdf(x)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, y, 'k-', linewidth=2)

# Fill regions
ax.fill_between(x, y, where=(x >= -1) & (x <= 1), alpha=0.3, color='blue', label='68.3% (1 SD)')
ax.fill_between(x, y, where=((x >= -2) & (x < -1)) | ((x > 1) & (x <= 2)), alpha=0.3, color='green', label='+ 27.2% (2 SD)')
ax.fill_between(x, y, where=((x >= -3) & (x < -2)) | ((x > 2) & (x <= 3)), alpha=0.3, color='orange', label='+ 4.3% (3 SD)')

# Mark standard deviations
for sd_val in [-3, -2, -1, 0, 1, 2, 3]:
    label = f'{sd_val} SD' if sd_val != 0 else 'mean'
    ax.axvline(sd_val, color='gray', linestyle=':', alpha=0.5)
    ax.text(sd_val, -0.02, label, ha='center', fontsize=8)

ax.set_xlabel('Standard Deviations from Mean')
ax.set_ylabel('Probability Density')
ax.set_title('Normal Distribution and the Empirical Rule')
ax.legend()
ax.set_xlim(-4, 4)
plt.tight_layout()
plt.show()

### 4.2 Distributions Used in Bioinformatics

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
np.random.seed(42)

# 1. Normal -- log-transformed expression data
data_norm = np.random.normal(10, 2, 2000)
axes[0, 0].hist(data_norm, bins=40, density=True, alpha=0.7, color='steelblue')
axes[0, 0].set_title('Normal\n(log-expression values)')

# 2. Poisson -- read counts (simple model)
data_pois = np.random.poisson(lam=20, size=2000)
axes[0, 1].hist(data_pois, bins=range(0, 45), density=True, alpha=0.7, color='coral')
axes[0, 1].set_title('Poisson (lambda=20)\n(simple read count model)')

# 3. Negative Binomial -- RNA-seq counts (accounts for overdispersion)
data_nb = np.random.negative_binomial(n=5, p=5/(5+20), size=2000)
axes[0, 2].hist(data_nb, bins=40, density=True, alpha=0.7, color='mediumpurple')
axes[0, 2].set_title('Negative Binomial\n(RNA-seq counts, DESeq2 model)')

# 4. Binomial -- variant allele frequency
data_binom = np.random.binomial(n=100, p=0.5, size=2000)
axes[1, 0].hist(data_binom, bins=30, density=True, alpha=0.7, color='darkseagreen')
axes[1, 0].set_title('Binomial (n=100, p=0.5)\n(variant allele counts)')

# 5. Uniform -- p-values under the null
data_unif = np.random.uniform(0, 1, 2000)
axes[1, 1].hist(data_unif, bins=20, density=True, alpha=0.7, color='goldenrod')
axes[1, 1].set_title('Uniform(0,1)\n(p-values when H0 is true)')

# 6. Chi-squared -- goodness of fit
data_chi2 = np.random.chisquare(df=5, size=2000)
axes[1, 2].hist(data_chi2, bins=40, density=True, alpha=0.7, color='tomato')
axes[1, 2].set_title('Chi-squared (df=5)\n(goodness-of-fit test)')

for ax in axes.flat:
    ax.set_ylabel('Density')

plt.suptitle('Distributions in Bioinformatics', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Why Negative Binomial for RNA-seq?**

The Poisson distribution assumes variance = mean. In real RNA-seq data, the variance is typically *larger* than the mean (overdispersion) due to biological variability between replicates. The Negative Binomial distribution adds a dispersion parameter to model this extra variability. This is why DESeq2 and edgeR use NB, not Poisson.

---

## 5. The Central Limit Theorem (CLT)

One of the most important theorems in statistics:

> Regardless of the original distribution, the **distribution of sample means** approaches a normal distribution as the sample size increases.

This is why t-tests and ANOVA work even when individual observations are not perfectly normal -- they test the *mean*, and the CLT guarantees that sample means are approximately normal for moderate n.

The standard error of the mean (SEM) quantifies how precisely we know the sample mean:

**SEM = SD / sqrt(n)**

More samples = smaller SEM = more precise estimate of the true mean.

In [ ]:
# Demonstrate the CLT with a non-normal population
np.random.seed(42)

# Start with a VERY non-normal distribution (exponential -- skewed)
population = np.random.exponential(scale=5, size=100000)

# Take many samples of different sizes and record their means
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].hist(population, bins=50, density=True, color='gray', alpha=0.7)
axes[0].set_title('Population\n(Exponential, highly skewed)')
axes[0].set_xlabel('Value')

for i, n_samples in enumerate([5, 30, 100]):
    sample_means = [np.mean(np.random.choice(population, size=n_samples)) for _ in range(2000)]
    axes[i + 1].hist(sample_means, bins=40, density=True, color='steelblue', alpha=0.7)
    axes[i + 1].set_title(f'Sample Means (n={n_samples})\nSD of means = {np.std(sample_means):.2f}')
    axes[i + 1].set_xlabel('Sample Mean')

plt.suptitle('Central Limit Theorem: Sample means become normal', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Population SD: {np.std(population):.2f}")
print(f"Theoretical SEM for n=30: {np.std(population) / np.sqrt(30):.2f}")
print(f"Observed SD of sample means (n=30): {np.std([np.mean(np.random.choice(population, 30)) for _ in range(2000)]):.2f}")

---

## 6. Hypothesis Testing

### The Framework

1. **State hypotheses**: H0 (null: no effect) and H1 (alternative: there IS an effect)
2. **Choose significance level**: alpha (typically 0.05)
3. **Calculate test statistic**: t-value, z-score, chi-square, etc.
4. **Calculate p-value**: probability of seeing data this extreme if H0 is true
5. **Decide**: if p < alpha, reject H0

### What the p-value IS and IS NOT

**P-value IS**: the probability of observing results at least as extreme as what we measured, *assuming the null hypothesis is true*.

**P-value is NOT**:
- The probability that H0 is true
- The probability of a false positive
- A measure of effect size or biological importance

In [ ]:
# Example: Is BRCA1 expression different between tumor and normal tissue?
np.random.seed(42)

# Normal tissue: 30 samples, mean expression ~10
normal_tissue = np.random.normal(loc=10, scale=2, size=30)

# Tumor tissue: 30 samples, mean expression ~12 (upregulated)
tumor_tissue = np.random.normal(loc=12, scale=2.5, size=30)

# Two-sample t-test (independent samples, two-sided)
t_stat, p_value = stats.ttest_ind(normal_tissue, tumor_tissue)

print("=== BRCA1 Differential Expression Test ===")
print(f"Normal tissue: mean = {np.mean(normal_tissue):.2f}, SD = {np.std(normal_tissue, ddof=1):.2f}, n = {len(normal_tissue)}")
print(f"Tumor tissue:  mean = {np.mean(tumor_tissue):.2f}, SD = {np.std(tumor_tissue, ddof=1):.2f}, n = {len(tumor_tissue)}")
print(f"\nFold change:   {np.mean(tumor_tissue) / np.mean(normal_tissue):.2f}x")
print(f"Log2 FC:       {np.log2(np.mean(tumor_tissue) / np.mean(normal_tissue)):.2f}")
print(f"\nT-statistic:   {t_stat:.3f}")
print(f"P-value:       {p_value:.2e}")
print(f"\nConclusion at alpha=0.05: {'Reject H0 -- significant difference' if p_value < 0.05 else 'Fail to reject H0'}")

In [ ]:
# Visualize what the p-value means
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: the data
axes[0].hist(normal_tissue, bins=15, alpha=0.6, color='steelblue', label='Normal', density=True)
axes[0].hist(tumor_tissue, bins=15, alpha=0.6, color='firebrick', label='Tumor', density=True)
axes[0].axvline(np.mean(normal_tissue), color='steelblue', linestyle='--', linewidth=2)
axes[0].axvline(np.mean(tumor_tissue), color='firebrick', linestyle='--', linewidth=2)
axes[0].set_xlabel('Expression Level')
axes[0].set_ylabel('Density')
axes[0].set_title('Expression Distribution')
axes[0].legend()

# Right: t-distribution showing the p-value area
df = len(normal_tissue) + len(tumor_tissue) - 2
x = np.linspace(-5, 5, 1000)
y = stats.t.pdf(x, df=df)
axes[1].plot(x, y, 'k-', linewidth=2)
axes[1].fill_between(x, y, where=(x <= -abs(t_stat)) | (x >= abs(t_stat)),
                     alpha=0.3, color='red', label=f'p-value = {p_value:.2e}')
axes[1].axvline(t_stat, color='red', linestyle='--', label=f't = {t_stat:.2f}')
axes[1].axvline(-t_stat, color='red', linestyle='--')
axes[1].set_xlabel('t-statistic')
axes[1].set_ylabel('Probability Density')
axes[1].set_title(f't-Distribution (df={df})')
axes[1].legend()

plt.tight_layout()
plt.show()

### Types of Errors

|  | H0 True (no real effect) | H0 False (real effect exists) |
|---|---|---|
| **Reject H0** | **Type I Error** (false positive, rate = alpha) | Correct! (true positive, rate = Power) |
| **Fail to reject H0** | Correct! (true negative) | **Type II Error** (false negative, rate = beta) |

- **alpha** = P(Type I error) = P(reject H0 | H0 true). We set this (typically 0.05).
- **beta** = P(Type II error) = P(fail to reject H0 | H0 false)
- **Power** = 1 - beta = P(reject H0 | H0 false). We want this to be high (> 0.8).

---

## 7. Choosing the Right Test

### Parametric vs. Non-Parametric

**Parametric tests** (t-test, ANOVA) assume the data follows a specific distribution (usually normal). They are more powerful when assumptions are met.

**Non-parametric tests** (Mann-Whitney, Kruskal-Wallis) make no distributional assumptions. They work on ranks. Use them when data is clearly non-normal, ordinal, or has small sample sizes.

| Question | Parametric | Non-Parametric |
|----------|-----------|----------------|
| Compare 2 independent groups | Independent t-test | Mann-Whitney U |
| Compare 2 paired groups | Paired t-test | Wilcoxon signed-rank |
| Compare 3+ groups | One-way ANOVA | Kruskal-Wallis |
| Association between 2 continuous vars | Pearson correlation | Spearman correlation |
| Association between 2 categorical vars | Chi-square test | Fisher's exact test |

In [ ]:
# Testing for normality -- should we use parametric or non-parametric?
np.random.seed(42)

# Normal data
normal_data = np.random.normal(10, 2, 50)

# Non-normal data (e.g., read counts are often right-skewed)
skewed_data = np.random.exponential(5, 50)

# Shapiro-Wilk test for normality
stat_n, p_n = stats.shapiro(normal_data)
stat_s, p_s = stats.shapiro(skewed_data)

print("Shapiro-Wilk Normality Test:")
print(f"  Normal data:  W = {stat_n:.4f}, p = {p_n:.4f} -> {'Normal' if p_n > 0.05 else 'Not normal'}")
print(f"  Skewed data:  W = {stat_s:.4f}, p = {p_s:.4f} -> {'Normal' if p_s > 0.05 else 'Not normal'}")
print("\nNote: Always combine with visual inspection! Shapiro-Wilk is overly")
print("sensitive with large samples and may reject slight deviations from normality.")

In [ ]:
# Comparing parametric and non-parametric tests on the same data
np.random.seed(42)

# Scenario: Comparing protein levels between two patient groups
group_A = np.random.exponential(5, 25)  # Right-skewed (non-normal)
group_B = np.random.exponential(8, 25)  # Right-skewed, higher

# Parametric: t-test (assumes normality)
t_stat, t_pval = stats.ttest_ind(group_A, group_B)

# Non-parametric: Mann-Whitney U (works on ranks, no normality assumption)
u_stat, mw_pval = stats.mannwhitneyu(group_A, group_B, alternative='two-sided')

print("Comparing protein levels between two patient groups (skewed data):")
print(f"  Group A: median = {np.median(group_A):.2f}, mean = {np.mean(group_A):.2f}")
print(f"  Group B: median = {np.median(group_B):.2f}, mean = {np.mean(group_B):.2f}")
print(f"\n  t-test p-value:       {t_pval:.4f}")
print(f"  Mann-Whitney p-value: {mw_pval:.4f}")
print(f"\n  For skewed data, prefer the Mann-Whitney U test.")

In [ ]:
# Paired t-test: before/after treatment on the SAME patients
np.random.seed(42)

# 20 patients measured before and after treatment
before = np.random.normal(100, 15, 20)  # Baseline tumor marker
after = before - np.random.normal(10, 8, 20)  # Treatment reduces marker

# Paired test (accounts for individual baseline differences)
t_paired, p_paired = stats.ttest_rel(before, after)

# Wrong approach: unpaired test (ignores pairing)
t_unpaired, p_unpaired = stats.ttest_ind(before, after)

print("Tumor marker levels before/after treatment (paired design):")
print(f"  Before: mean = {np.mean(before):.1f}")
print(f"  After:  mean = {np.mean(after):.1f}")
print(f"  Mean difference: {np.mean(before - after):.1f}")
print(f"\n  Paired t-test:   p = {p_paired:.4f}  (CORRECT -- uses the pairing)")
print(f"  Unpaired t-test: p = {p_unpaired:.4f}  (WRONG -- ignores pairing, less power)")

In [ ]:
# ANOVA: comparing 3+ groups
np.random.seed(42)

# Gene expression across 4 tissue types
brain = np.random.normal(12, 2, 25)
liver = np.random.normal(8, 2, 25)
kidney = np.random.normal(10, 2, 25)
heart = np.random.normal(9, 2, 25)

# One-way ANOVA
f_stat, anova_p = stats.f_oneway(brain, liver, kidney, heart)

print("Gene expression across tissue types:")
print(f"  Brain:  {np.mean(brain):.2f} +/- {np.std(brain, ddof=1):.2f}")
print(f"  Liver:  {np.mean(liver):.2f} +/- {np.std(liver, ddof=1):.2f}")
print(f"  Kidney: {np.mean(kidney):.2f} +/- {np.std(kidney, ddof=1):.2f}")
print(f"  Heart:  {np.mean(heart):.2f} +/- {np.std(heart, ddof=1):.2f}")
print(f"\n  ANOVA: F = {f_stat:.2f}, p = {anova_p:.2e}")
print(f"  Conclusion: {'At least one group differs' if anova_p < 0.05 else 'No significant difference'}")

# Non-parametric alternative: Kruskal-Wallis
kw_stat, kw_p = stats.kruskal(brain, liver, kidney, heart)
print(f"\n  Kruskal-Wallis: H = {kw_stat:.2f}, p = {kw_p:.2e}")

In [ ]:
# Correlation: Pearson vs Spearman
np.random.seed(42)

# Two genes with correlated expression
gene_x = np.random.normal(10, 3, 50)
gene_y = 0.7 * gene_x + np.random.normal(0, 2, 50)

# Pearson (linear relationship, assumes normality)
r_pearson, p_pearson = stats.pearsonr(gene_x, gene_y)

# Spearman (monotonic relationship, rank-based, no normality assumption)
r_spearman, p_spearman = stats.spearmanr(gene_x, gene_y)

print("Gene co-expression correlation:")
print(f"  Pearson r = {r_pearson:.3f}, p = {p_pearson:.2e}")
print(f"  Spearman rho = {r_spearman:.3f}, p = {p_spearman:.2e}")

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(gene_x, gene_y, alpha=0.6, color='steelblue')
z = np.polyfit(gene_x, gene_y, 1)
ax.plot(np.sort(gene_x), np.polyval(z, np.sort(gene_x)), 'r-', linewidth=2)
ax.set_xlabel('Gene X expression')
ax.set_ylabel('Gene Y expression')
ax.set_title(f'Co-expression (Pearson r = {r_pearson:.2f}, p = {p_pearson:.1e})')
plt.tight_layout()
plt.show()

In [ ]:
# Chi-square and Fisher's exact test: categorical data
# Example: Is a mutation associated with drug response?

# Contingency table
#                Responder  Non-responder
# Mutation+         35          15
# Mutation-         20          30

observed = np.array([[35, 15],
                     [20, 30]])

# Chi-square test
chi2, chi2_p, dof, expected = stats.chi2_contingency(observed)

print("Mutation vs Drug Response:")
print(f"  Observed table:")
print(f"                 Responder  Non-responder")
print(f"  Mutation+:       {observed[0,0]:5d}        {observed[0,1]:5d}")
print(f"  Mutation-:       {observed[1,0]:5d}        {observed[1,1]:5d}")
print(f"\n  Chi-square = {chi2:.2f}, p = {chi2_p:.4f}")

# Fisher's exact test (better for small samples)
odds_ratio, fisher_p = stats.fisher_exact(observed)
print(f"  Fisher's exact: OR = {odds_ratio:.2f}, p = {fisher_p:.4f}")
print(f"\n  Interpretation: Patients with the mutation are {odds_ratio:.1f}x more likely to respond.")

---

## 8. Multiple Testing Correction

This is one of the most critical concepts in genomics. When you test thousands of genes simultaneously, the chance of false positives explodes.

**The problem**: testing 20,000 genes at alpha = 0.05 produces ~1,000 false positives (20,000 x 0.05), even when NO genes are truly differentially expressed.

### Two main approaches

**Bonferroni correction** (controls Family-Wise Error Rate)
- Adjusted threshold: alpha / n_tests
- Very conservative: minimizes false positives but misses many real effects
- Use when false positives are very costly (e.g., drug targets)

**Benjamini-Hochberg (BH) procedure** (controls False Discovery Rate)
- FDR = expected proportion of false discoveries among all discoveries
- Less conservative: better balance of false positives and false negatives
- **Standard in genomics** -- DESeq2, edgeR, and most tools report FDR-adjusted p-values

In [ ]:
from statsmodels.stats.multitest import multipletests

np.random.seed(42)

# Simulate a genome-wide experiment
n_genes = 10000
n_true_de = 200  # Only 200 genes are truly differentially expressed (2%)

# Generate p-values
# Non-DE genes: p-values uniform between 0 and 1 (null distribution)
# True DE genes: small p-values (drawn from Beta distribution)
p_null = np.random.uniform(0, 1, n_genes - n_true_de)
p_de = np.random.beta(0.3, 10, n_true_de)  # Concentrated near 0

all_pvalues = np.concatenate([p_null, p_de])
is_truly_de = np.concatenate([np.zeros(n_genes - n_true_de), np.ones(n_true_de)]).astype(bool)

# No correction
sig_none = all_pvalues < 0.05

# Bonferroni correction
reject_bonf, pvals_bonf, _, _ = multipletests(all_pvalues, alpha=0.05, method='bonferroni')

# Benjamini-Hochberg FDR
reject_bh, pvals_bh, _, _ = multipletests(all_pvalues, alpha=0.05, method='fdr_bh')

print(f"Genome-wide DE analysis: {n_genes} genes tested, {n_true_de} truly DE")
print(f"{'Method':<25} {'Called sig':>10} {'True pos':>10} {'False pos':>10} {'False neg':>10} {'FDR':>8}")
print("-" * 80)

for name, sig in [('No correction', sig_none), ('Bonferroni', reject_bonf), ('BH (FDR)', reject_bh)]:
    tp = np.sum(sig & is_truly_de)
    fp = np.sum(sig & ~is_truly_de)
    fn = np.sum(~sig & is_truly_de)
    total_sig = np.sum(sig)
    fdr = fp / total_sig if total_sig > 0 else 0
    print(f"{name:<25} {total_sig:>10} {tp:>10} {fp:>10} {fn:>10} {fdr:>7.1%}")

In [ ]:
# Visualize the effect of multiple testing correction
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# P-value histogram ("tower" at left = true DE genes)
axes[0].hist(all_pvalues, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(0.05, color='red', linestyle='--', label='alpha = 0.05')
axes[0].set_xlabel('P-value')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Raw P-value Distribution')
axes[0].legend()

# Adjusted p-values comparison
sort_idx = np.argsort(all_pvalues)
rank = np.arange(1, len(all_pvalues) + 1)

axes[1].plot(rank, np.sort(all_pvalues), 'b-', alpha=0.5, label='Raw p-value')
axes[1].plot(rank, np.sort(pvals_bh), 'r-', alpha=0.5, label='BH adjusted')
axes[1].axhline(0.05, color='gray', linestyle='--')
axes[1].set_xlabel('Rank')
axes[1].set_ylabel('P-value')
axes[1].set_title('Raw vs BH-adjusted P-values')
axes[1].set_xlim(0, 500)  # Zoom into the significant range
axes[1].legend()

# Volcano-style plot showing true vs false discoveries
neg_log10_p = -np.log10(all_pvalues)
colors = np.where(is_truly_de & reject_bh, 'green',     # True positive
         np.where(~is_truly_de & reject_bh, 'red',       # False positive
         np.where(is_truly_de & ~reject_bh, 'orange',    # False negative
         'gray')))                                         # True negative

axes[2].scatter(range(len(all_pvalues)), neg_log10_p, c=colors, s=3, alpha=0.5)
axes[2].axhline(-np.log10(0.05), color='gray', linestyle='--')
axes[2].set_xlabel('Gene index')
axes[2].set_ylabel('-log10(p-value)')
axes[2].set_title('Outcome per gene (BH FDR < 0.05)')

# Manual legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='green', markersize=8, label='True positive'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=8, label='False positive'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='orange', markersize=8, label='False negative'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=8, label='True negative'),
]
axes[2].legend(handles=legend_elements, fontsize=8, loc='upper right')

plt.tight_layout()
plt.show()

### Practical Guidance

| Situation | Recommended correction | Typical threshold |
|-----------|----------------------|------------------|
| RNA-seq DE analysis | BH (FDR) | FDR < 0.05 |
| Exploratory analysis | BH (FDR) | FDR < 0.10 |
| GWAS (very many tests) | Bonferroni | p < 5e-8 |
| Drug target validation (few tests, high stakes) | Bonferroni | p < 0.05 / n |
| Gene set enrichment | BH (FDR) | FDR < 0.05 |

---

## 9. Effect Size -- Statistical vs. Biological Significance

A p-value tells you whether an effect is *unlikely to be due to chance*, but it says nothing about *how large* the effect is. With enough samples, even a tiny, biologically meaningless difference becomes statistically significant.

**Always report effect sizes alongside p-values.**

Common effect size measures in bioinformatics:
- **Log2 fold change**: standard for differential expression
- **Cohen's d**: standardized mean difference (mean_A - mean_B) / pooled_SD
- **Odds ratio**: for categorical data (e.g., mutation vs. outcome)
- **Correlation coefficient (r)**: for continuous associations

In [ ]:
# Demonstration: statistical significance != biological significance
np.random.seed(42)

# Tiny effect but huge sample size -> significant p-value
n_large = 10000
group1 = np.random.normal(10.00, 2, n_large)
group2 = np.random.normal(10.05, 2, n_large)  # Only 0.05 difference!

t_stat_large, p_large = stats.ttest_ind(group1, group2)
effect_size_large = (np.mean(group2) - np.mean(group1)) / np.sqrt((np.var(group1, ddof=1) + np.var(group2, ddof=1)) / 2)

# Large effect but small sample size -> may not be significant
n_small = 5
group3 = np.random.normal(10, 2, n_small)
group4 = np.random.normal(14, 2, n_small)  # 4-unit difference!

t_stat_small, p_small = stats.ttest_ind(group3, group4)
effect_size_small = (np.mean(group4) - np.mean(group3)) / np.sqrt((np.var(group3, ddof=1) + np.var(group4, ddof=1)) / 2)

print("Scenario 1: Tiny effect, huge n")
print(f"  Mean diff: {np.mean(group2) - np.mean(group1):.3f}")
print(f"  Cohen's d: {effect_size_large:.3f} (negligible)")
print(f"  p-value:   {p_large:.4f}")
print(f"  n per group: {n_large}")

print(f"\nScenario 2: Large effect, tiny n")
print(f"  Mean diff: {np.mean(group4) - np.mean(group3):.3f}")
print(f"  Cohen's d: {effect_size_small:.3f} (large)")
print(f"  p-value:   {p_small:.4f}")
print(f"  n per group: {n_small}")

print("\nLesson: Always consider BOTH p-value AND effect size!")
print("In genomics, we often filter for |log2FC| > 1 AND FDR < 0.05.")

---

## 10. Power Analysis -- Plan Before You Experiment

**Statistical power** = probability of detecting a real effect when it exists (1 - beta).

Power depends on four factors (know any three, solve for the fourth):
1. **Sample size (n)**: larger n = more power
2. **Effect size (d)**: larger effect = easier to detect
3. **Significance level (alpha)**: larger alpha = more power (but more false positives)
4. **Variance**: less noise = more power

**Always do power analysis BEFORE running an experiment** to ensure you have enough samples to detect the effect you care about.

In [ ]:
from statsmodels.stats.power import TTestIndPower

power_analysis = TTestIndPower()

# How many samples do I need to detect a medium effect (Cohen's d = 0.5)?
effect_sizes = [0.2, 0.5, 0.8]  # Small, medium, large
labels = ['Small (d=0.2)', 'Medium (d=0.5)', 'Large (d=0.8)']

print("Required samples per group (alpha=0.05, power=0.80):")
print("-" * 50)
for es, label in zip(effect_sizes, labels):
    n = power_analysis.solve_power(effect_size=es, alpha=0.05, power=0.80, alternative='two-sided')
    print(f"  {label:<20} -> n = {int(np.ceil(n)):>4} per group")

In [ ]:
# Power curves: how does power change with sample size?
sample_sizes = np.arange(5, 200, 1)

fig, ax = plt.subplots(figsize=(10, 5))

for es, label, color in zip([0.2, 0.5, 0.8],
                            ['Small (d=0.2)', 'Medium (d=0.5)', 'Large (d=0.8)'],
                            ['steelblue', 'firebrick', 'darkgreen']):
    powers = [power_analysis.power(effect_size=es, nobs1=n, alpha=0.05, ratio=1.0, alternative='two-sided') 
              for n in sample_sizes]
    ax.plot(sample_sizes, powers, label=label, linewidth=2, color=color)

ax.axhline(0.80, color='gray', linestyle='--', label='80% power target')
ax.set_xlabel('Sample Size per Group')
ax.set_ylabel('Statistical Power')
ax.set_title('Power Analysis: How Many Samples Do You Need?')
ax.legend()
ax.set_ylim(0, 1.02)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Interpretation: With n=30 per group, you have ~80% power to detect")
print("a medium effect (d=0.5), but only ~15% power for a small effect (d=0.2).")
print("This is why RNA-seq experiments with only 2-3 replicates miss many real DE genes!")

---

## 11. Worked Example: Complete Differential Expression Analysis

Let us walk through a complete analysis mimicking an RNA-seq experiment, from data generation to final results.

In [ ]:
np.random.seed(42)

# Simulate RNA-seq-like data for 5000 genes, 6 samples (3 control, 3 treatment)
n_genes = 5000
n_control = 3
n_treatment = 3
n_truly_de = 250  # 5% of genes are truly DE

# Base expression levels (log-normal, as in real RNA-seq)
base_expr = np.random.lognormal(mean=3, sigma=1.5, size=n_genes)

# Generate expression for each sample
control_data = np.array([base_expr * np.random.lognormal(0, 0.3, n_genes) for _ in range(n_control)]).T
treatment_data = np.array([base_expr * np.random.lognormal(0, 0.3, n_genes) for _ in range(n_treatment)]).T

# Add real differential expression for the first 250 genes
de_effects = np.concatenate([
    np.random.choice([-1, 1], n_truly_de) * np.random.uniform(1.5, 4, n_truly_de),  # log2 FC between 1.5 and 4
    np.zeros(n_genes - n_truly_de)
])
treatment_data[:, :] *= 2 ** de_effects[:, np.newaxis]  # Apply fold changes

# Log2 transform
control_log2 = np.log2(control_data + 1)
treatment_log2 = np.log2(treatment_data + 1)

gene_names = [f"Gene_{i:04d}" for i in range(n_genes)]

print(f"Simulated dataset: {n_genes} genes, {n_control} control + {n_treatment} treatment samples")
print(f"Truly DE genes: {n_truly_de} ({100*n_truly_de/n_genes:.0f}%)")
print(f"Control data shape: {control_log2.shape}")
print(f"Treatment data shape: {treatment_log2.shape}")

In [ ]:
# Step 1: Test each gene with a t-test
pvalues = []
log2_fold_changes = []
mean_expressions = []

for i in range(n_genes):
    ctrl = control_log2[i, :]
    treat = treatment_log2[i, :]
    
    # Two-sample t-test
    _, pval = stats.ttest_ind(ctrl, treat)
    pvalues.append(pval)
    
    # Log2 fold change
    l2fc = np.mean(treat) - np.mean(ctrl)
    log2_fold_changes.append(l2fc)
    
    # Mean expression
    mean_expressions.append(np.mean(np.concatenate([ctrl, treat])))

pvalues = np.array(pvalues)
log2_fold_changes = np.array(log2_fold_changes)
mean_expressions = np.array(mean_expressions)

# Step 2: Multiple testing correction (BH/FDR)
reject_fdr, fdr_values, _, _ = multipletests(pvalues, alpha=0.05, method='fdr_bh')

# Step 3: Define significant DE genes (|log2FC| > 1 AND FDR < 0.05)
is_significant = reject_fdr & (np.abs(log2_fold_changes) > 1)

print(f"Results summary:")
print(f"  Genes with raw p < 0.05:       {np.sum(pvalues < 0.05)}")
print(f"  Genes with FDR < 0.05:         {np.sum(reject_fdr)}")
print(f"  Genes with FDR < 0.05 & |FC|>2:{np.sum(is_significant)}")
print(f"  (Truly DE genes: {n_truly_de})")

In [ ]:
# Step 4: Volcano plot
is_truly_de_arr = np.zeros(n_genes, dtype=bool)
is_truly_de_arr[:n_truly_de] = True

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Volcano plot
neg_log10_fdr = -np.log10(fdr_values + 1e-300)  # Add tiny value to avoid log(0)

colors = np.full(n_genes, 'gray')
colors[is_significant & (log2_fold_changes > 0)] = 'firebrick'
colors[is_significant & (log2_fold_changes < 0)] = 'steelblue'

axes[0].scatter(log2_fold_changes, neg_log10_fdr, c=colors, s=5, alpha=0.5)
axes[0].axhline(-np.log10(0.05), color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(-1, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(1, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Log2 Fold Change')
axes[0].set_ylabel('-Log10(FDR)')
axes[0].set_title(f'Volcano Plot\n({np.sum(is_significant)} significant DE genes)')

# Count annotations
n_up = np.sum(is_significant & (log2_fold_changes > 0))
n_down = np.sum(is_significant & (log2_fold_changes < 0))
axes[0].text(0.02, 0.98, f'Up: {n_up}\nDown: {n_down}', transform=axes[0].transAxes,
             verticalalignment='top', fontsize=11, fontweight='bold')

# MA plot
axes[1].scatter(mean_expressions, log2_fold_changes, c=colors, s=5, alpha=0.5)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].axhline(-1, color='gray', linestyle='--', alpha=0.5)
axes[1].axhline(1, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Mean Expression (log2)')
axes[1].set_ylabel('Log2 Fold Change')
axes[1].set_title('MA Plot')

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate our analysis: how well did we recover the truly DE genes?
tp = np.sum(is_significant & is_truly_de_arr)
fp = np.sum(is_significant & ~is_truly_de_arr)
fn = np.sum(~is_significant & is_truly_de_arr)
tn = np.sum(~is_significant & ~is_truly_de_arr)

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
observed_fdr = fp / (tp + fp) if (tp + fp) > 0 else 0

print("Performance evaluation (since we know ground truth):")
print(f"  True Positives:  {tp} (correctly identified DE genes)")
print(f"  False Positives: {fp} (non-DE genes incorrectly called significant)")
print(f"  False Negatives: {fn} (DE genes we missed)")
print(f"  True Negatives:  {tn}")
print(f"\n  Precision (1-FDR): {precision:.1%} (of our discoveries, this % are real)")
print(f"  Recall (sensitivity): {recall:.1%} (of true DE genes, this % were found)")
print(f"  Observed FDR: {observed_fdr:.1%} (target was 5%)")
print(f"\nNote: With only 3 replicates, recall is low. More replicates = more power.")

---

## Summary: Key Formulas and Concepts

### Descriptive Statistics
- **Mean**: x-bar = sum(x_i) / n
- **Variance**: s^2 = sum((x_i - x-bar)^2) / (n-1)
- **Standard Deviation**: s = sqrt(variance)
- **Standard Error**: SEM = s / sqrt(n)
- **Z-score**: z = (x - mean) / SD

### P-value Interpretation
| P-value | Evidence against H0 |
|---------|--------------------|
| p < 0.001 | Very strong |
| p < 0.01 | Strong |
| p < 0.05 | Moderate (typical threshold) |
| p >= 0.05 | Insufficient to reject H0 |

### Key Takeaways

1. **Sample is not population**: statistics are estimates with uncertainty
2. **P-value**: probability of data given H0, NOT probability that H0 is true
3. **Multiple testing correction is mandatory** when testing many genes (use BH/FDR)
4. **Effect size matters**: statistical significance is not biological significance
5. **Power analysis**: plan sample sizes BEFORE the experiment
6. **Choose the right test**: parametric if data is normal, non-parametric otherwise
7. **Always visualize your data** before running tests

---

## Exercises

### Exercise 1: Descriptive Statistics on Gene Expression

The following array represents log2 TPM values for a gene across 40 cancer samples:

```python
np.random.seed(123)
expression = np.concatenate([np.random.normal(8, 1.5, 35), [15, 16, 18, 20, 22]])
```

1. Calculate mean, median, SD, and IQR
2. Which measure of central tendency is more appropriate here? Why?
3. Identify outliers using the 1.5*IQR rule
4. Create a boxplot and histogram side by side

In [ ]:
# YOUR CODE HERE


### Exercise 2: Hypothesis Testing

A researcher measures TP53 expression in 25 tumor samples and 25 matched normal samples:

```python
np.random.seed(456)
normal = np.random.normal(6.5, 1.2, 25)
tumor = np.random.normal(4.8, 1.5, 25)
```

1. Test whether TP53 is significantly downregulated in tumors (use appropriate test)
2. First check normality of both groups
3. Calculate Cohen's d effect size
4. What would the minimum sample size need to be to detect this effect with 90% power?

In [ ]:
# YOUR CODE HERE


### Exercise 3: Multiple Testing Correction

You test 500 genes and obtain the following p-values (simulated):

```python
np.random.seed(789)
pvals = np.concatenate([np.random.beta(0.1, 5, 30), np.random.uniform(0, 1, 470)])
```

1. How many genes are significant at p < 0.05 without correction?
2. Apply Bonferroni correction -- how many survive?
3. Apply BH (FDR) correction -- how many survive?
4. Plot the p-value histogram and discuss its shape

In [ ]:
# YOUR CODE HERE


---

## Exercise Solutions

In [ ]:
# Solution 1: Descriptive Statistics
np.random.seed(123)
expression = np.concatenate([np.random.normal(8, 1.5, 35), [15, 16, 18, 20, 22]])

# 1. Statistics
mean_val = np.mean(expression)
median_val = np.median(expression)
sd_val = np.std(expression, ddof=1)
q25, q75 = np.percentile(expression, [25, 75])
iqr_val = q75 - q25

print(f"Mean:   {mean_val:.2f}")
print(f"Median: {median_val:.2f}")
print(f"SD:     {sd_val:.2f}")
print(f"IQR:    {iqr_val:.2f} (Q1={q25:.2f}, Q3={q75:.2f})")

# 2. Median is more appropriate because the outliers skew the mean upward
print(f"\nMedian is more appropriate: the high outliers pull the mean up by {mean_val - median_val:.2f}")

# 3. Outliers using 1.5*IQR rule
lower_fence = q25 - 1.5 * iqr_val
upper_fence = q75 + 1.5 * iqr_val
outliers = expression[(expression < lower_fence) | (expression > upper_fence)]
print(f"\nOutlier fences: [{lower_fence:.2f}, {upper_fence:.2f}]")
print(f"Outliers: {sorted(outliers)}")

# 4. Plots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot(expression, vert=True, patch_artist=True, boxprops=dict(facecolor='lightblue'))
axes[0].set_ylabel('Expression (log2 TPM)')
axes[0].set_title('Boxplot')

axes[1].hist(expression, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
axes[1].axvline(mean_val, color='red', linestyle='--', label=f'Mean: {mean_val:.1f}')
axes[1].axvline(median_val, color='green', linestyle='--', label=f'Median: {median_val:.1f}')
axes[1].set_xlabel('Expression (log2 TPM)')
axes[1].set_title('Histogram')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Solution 2: Hypothesis Testing
np.random.seed(456)
normal = np.random.normal(6.5, 1.2, 25)
tumor = np.random.normal(4.8, 1.5, 25)

# 2. Check normality
_, p_norm_normal = stats.shapiro(normal)
_, p_norm_tumor = stats.shapiro(tumor)
print(f"Normality tests:")
print(f"  Normal tissue: Shapiro p = {p_norm_normal:.4f} -> {'Normal' if p_norm_normal > 0.05 else 'Not normal'}")
print(f"  Tumor tissue:  Shapiro p = {p_norm_tumor:.4f} -> {'Normal' if p_norm_tumor > 0.05 else 'Not normal'}")

# 1. t-test (appropriate since both are normal)
t_stat, p_val = stats.ttest_ind(normal, tumor)
print(f"\nt-test: t = {t_stat:.3f}, p = {p_val:.2e}")
print(f"Conclusion: TP53 is {'significantly' if p_val < 0.05 else 'not significantly'} downregulated in tumors")

# 3. Cohen's d
pooled_sd = np.sqrt((np.var(normal, ddof=1) + np.var(tumor, ddof=1)) / 2)
cohens_d = (np.mean(normal) - np.mean(tumor)) / pooled_sd
print(f"\nCohen's d = {cohens_d:.2f} (large effect)")

# 4. Power analysis
from statsmodels.stats.power import TTestIndPower
pa = TTestIndPower()
n_needed = pa.solve_power(effect_size=cohens_d, alpha=0.05, power=0.90, alternative='two-sided')
print(f"Samples needed per group for 90% power: {int(np.ceil(n_needed))}")

In [ ]:
# Solution 3: Multiple Testing Correction
from statsmodels.stats.multitest import multipletests

np.random.seed(789)
pvals = np.concatenate([np.random.beta(0.1, 5, 30), np.random.uniform(0, 1, 470)])

# 1. No correction
n_raw = np.sum(pvals < 0.05)
print(f"Significant at p < 0.05 (no correction): {n_raw}")

# 2. Bonferroni
reject_bonf, _, _, _ = multipletests(pvals, alpha=0.05, method='bonferroni')
print(f"Significant after Bonferroni: {np.sum(reject_bonf)}")

# 3. BH (FDR)
reject_bh, _, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')
print(f"Significant after BH (FDR): {np.sum(reject_bh)}")

# 4. P-value histogram
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(pvals, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(0.05, color='red', linestyle='--', label='alpha = 0.05')
ax.set_xlabel('P-value')
ax.set_ylabel('Frequency')
ax.set_title('P-value Distribution')
ax.legend()
plt.tight_layout()
plt.show()

print("\nThe spike near 0 represents the truly DE genes (low p-values).")
print("The flat portion represents null genes (uniform p-values under H0).")
print("This shape is exactly what we expect: most genes are not DE.")

---

[< Previous: R Fundamentals for Bioinformatics](../05_R_Basics/01_r_fundamentals.ipynb) | [Tier 0: Computational Foundations Overview](../README.md) | [Next: Probability >](../07_Probability_and_Statistics_Python/01_probability.ipynb)

---